In [ ]:
# Pregunta: ¿Qué capacidad diaria debe operar el centro para cumplir la meta de servicio, controlar el riesgo y respetar el presupuesto?

from pathlib import Path
import json
import sys

import pandas as pd

root = Path.cwd().parent
sys.path.insert(0, str(root / 'src'))
from main import choose_policy, evaluate, policy_record


In [ ]:
policies = pd.read_csv(root / 'data' / 'capacity_policies.csv')
scenarios = pd.read_csv(root / 'data' / 'demand_scenarios.csv')

policies


In [ ]:
# Los escenarios hacen visible la incertidumbre que la política debe soportar.

scenarios


In [ ]:
comparison = evaluate(policies, scenarios)
comparison = comparison.sort_values('daily_cost').reset_index(drop=True)
comparison


In [ ]:
selected_policy = choose_policy(comparison)

scenario_effects = scenarios.assign(
    served=scenarios.demand.clip(upper=selected_policy.capacity),
    service_rate=lambda frame: frame.served / frame.demand,
)
scenario_effects


In [ ]:
decision = policy_record(selected_policy)

comparison.to_csv(root / 'submission' / 'capacity_policy_comparison.csv', index=False)
with open(root / 'submission' / 'capacity_policy_decision.json', 'w') as file:
    json.dump(decision, file, indent=2, ensure_ascii=False)

print(json.dumps(decision, indent=2, ensure_ascii=False))
